# Lab 4B: Build a tiny MCP server and call it (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 4 | about 10 minutes

- Write a small MCP server (two tools, one resource, one prompt) that runs over stdio
- Spawn it from a client, list its tools and call them
- No LLM and no embeddings needed

## 1. The server
MCP servers expose three primitives: **tools** (actions), **resources** (readable data) and **prompts** (reusable templates).

In [ ]:
import os, sys, json, asyncio, concurrent.futures
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = '''
import re, glob, ast, operator
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("tiny-company-tools")

def _words(s): return set(re.findall(r"[a-z0-9]{3,}", s.lower()))
DOCS = []
for f in sorted(glob.glob("data/corpus/*.txt")):
    head, body = open(f, encoding="utf-8").read().split("\\n\\n", 1)
    DOCS.append((head.splitlines()[0].replace("title: ", ""), body.strip()))

@mcp.tool()
def search_docs(query: str, k: int = 2) -> str:
    """Keyword-overlap search over the company documents; returns top-k titles with a snippet."""
    q = _words(query)
    ranked = sorted(DOCS, key=lambda d: -len(q & _words(d[0] + " " + d[1])))[:k]
    return "\\n".join(f"[{t}] {b[:200]}..." for t, b in ranked)

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}
@mcp.tool()
def calculator(expression: str) -> str:
    """Safe arithmetic only: numbers and + - * / ** %."""
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only plain arithmetic is allowed")
    return str(round(ev(ast.parse(expression, mode="eval").body), 4))

@mcp.resource("docs://count")
def doc_count() -> str:
    """How many documents the server can search."""
    return str(len(DOCS))

@mcp.prompt()
def cite_sources(question: str) -> str:
    """Prompt template: answer from documents and cite titles."""
    return f"Use search_docs to answer, and cite the document titles: {question}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
os.makedirs("artifacts", exist_ok=True)
open("artifacts/mcp_server.py", "w").write(SERVER)
print("wrote artifacts/mcp_server.py,", len(SERVER.splitlines()), "lines")

## 2. A client helper
The client spawns the server as a subprocess over stdio. The coroutine runs in its own thread and event loop, so it works in Jupyter and in a plain script.

In [ ]:
async def _session(fn):
    params = StdioServerParameters(command=sys.executable, args=["artifacts/mcp_server.py"])
    async with stdio_client(params, errlog=open(os.devnull, 'w')) as (r, w):  # hide server log noise
        async with ClientSession(r, w) as s:
            await s.initialize()
            return await fn(s)

def with_server(fn):
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, asyncio.wait_for(_session(fn), 25)).result()

async def list_tools(s):
    return [(t.name, (t.description or "").split("\n")[0]) for t in (await s.list_tools()).tools]

async def call(s, name, args):
    return (await s.call_tool(name, args)).content[0].text

for name, desc in with_server(list_tools):
    print(f"tool {name}: {desc}")

## 3. Call the tools

In [ ]:
async def demo(s):
    hit = await call(s, "search_docs", {"query": "can I work from Spain for a month remote work"})
    calc = await call(s, "calculator", {"expression": "210 * 3"})
    res = (await s.read_resource("docs://count")).contents[0].text
    return hit, calc, res

hit, calc, count = with_server(demo)
print("search_docs ->", hit.replace("\n", " | ")[:300])
print("calculator(210 * 3) ->", calc)
print("resource docs://count ->", count)

## 4. Using MCP from an agent, and security notes
In Lab 4A, replace the local `TOOLS` functions with calls through the client helper, e.g. `TOOLS = {"calculator": (lambda e: with_server(lambda s: call(s, "calculator", {"expression": e})), "...")}`.
The planner and loop stay the same, and the same server works with any MCP-capable client.

Security: authenticate clients, give each server the narrowest permissions, and log every tool call.

**Recap**
- An MCP server publishes tools, resources and prompts over a standard protocol (here stdio).
- Any MCP client can discover and call them without custom glue code.
- Next: agent design patterns.